> ### GA4_Data_Quality_Checks

### Step 1 — Read silver tables

In [0]:
%sql
-- This cell was converted to SQL for SQL warehouse compatibility
-- The DataFrames are replaced with CTEs that can be referenced in subsequent SQL cells

SELECT * FROM workspace.silver.ga4_silver
order by event_date DESC


In [0]:
%sql
SELECT* FROM workspace.silver.ga4_silver
where traffic_type = "internal"

### Step 2 — Row count checks

In [0]:
%sql
-- Row count
SELECT COUNT(*) as row_count FROM workspace.bronze.events
UNION ALL
SELECT COUNT(*) as row_count FROM workspace.silver.ga4_silver


### Step 3 — Null checks on critical columns

In [0]:
# Step 3 — Null checks on critical columns
from pyspark.sql.functions import col, count, when

def check_nulls(df, table_name):
    # Check null % for each column
    total = df.count()
    for c in df.columns:
        nulls = df.filter(col(c).isNull()).count()
        pct = round((nulls / total) * 100, 2) if total > 0 else 0
        status = "❌ FAIL" if pct > 5 else "✅ PASS"
        print(f"{status} | {table_name}.{c} | nulls: {nulls} ({pct}%)")

check_nulls(df_external, "ga4_silver")
check_nulls(df_internal, "ga4_silver_internal")
check_nulls(df_qurantine, "ga4_qurantine")

###  Step 4 — Duplicate check

In [0]:
# Step 4 — Duplicate check
def check_duplicates(df, table_name, key_cols):
    total = df.count()
    distinct = df.dropDuplicates(key_cols).count()
    dupes = total - distinct
    status = "❌ FAIL" if dupes > 0 else "✅ PASS"
    print(f"{status} | {table_name} | duplicates: {dupes}")

check_duplicates(df_external, "ga4_silver", 
                 ["event_timestamp", "event_name", "page_location", "country", "city"])
check_duplicates(df_internal, "ga4_silver_internal", 
                 ["event_timestamp", "event_name", "page_location", "country", "city"])
check_duplicates(df_internal, "ga4_qurantine", 
                 ["event_timestamp", "event_name", "page_location", "country", "city"])

### Step 5 — Traffic type check (no internal in external table)

In [0]:
# Step 5 — Traffic type check (no internal in external table)
def check_traffic_type(df, table_name, expected_exclusion):
    bad = df.filter(col("traffic_type") == expected_exclusion).count()
    status = "❌ FAIL" if bad > 0 else "✅ PASS"
    print(f"{status} | {table_name} | '{expected_exclusion}' rows found: {bad}")

check_traffic_type(df_external, "ga4_silver", "internal")

###  Step 6 — Freshness check

In [0]:
# Step 6 — Freshness check
from pyspark.sql.functions import max as spark_max

def check_freshness(df, table_name, timestamp_col, max_age_hours=24):
    from datetime import datetime, timezone
    latest = df.agg(spark_max(col(timestamp_col))).collect()[0][0]
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    age_hours = (now - latest.replace(tzinfo=None)).total_seconds() / 3600 if latest else float('inf')
    status = "❌ FAIL" if age_hours > max_age_hours else "✅ PASS"
    print(f"{status} | {table_name} | latest: {latest} | age: {round(age_hours, 1)}h")

check_freshness(df_external, "ga4_silver", "event_date")
check_freshness(df_internal, "ga4_silver_internal", "event_date")
check_freshness(df_qurantine, "ga4_qurantine", "event_date")

### Breakdown of Quarantine Failure Types

In [0]:
%sql

-- Breakdown of Quarantine Failure Types
SELECT
    CASE
        WHEN page_location IS NULL THEN 'missing page_location'
        WHEN page_title IS NULL    THEN 'missing page_title'
        WHEN event_timestamp IS NULL THEN 'missing event_timestamp'
        ELSE 'unknown'
    END AS failure_reason,
    COUNT(*) as count
FROM workspace.silver.ga4_quarantine
GROUP BY 1
ORDER BY count DESC;

### Check date range

In [0]:
%sql
-- Check date range across all silver tables
SELECT 'ga4_silver'          AS table_name, 
       MIN(try_cast(event_date AS DATE)) AS earliest, 
       MAX(try_cast(event_date AS DATE)) AS latest
FROM workspace.silver.ga4_silver


Check tables hixstory , event_log , system tables for more details 

In [0]:
df = spark.sql("""
    SELECT *
    FROM system.lakeflow.pipelines
    WHERE pipeline_id = '2f1ed3ce-6581-422f-a51d-218cd2092450'
""")
display(df)

In [0]:
%sql

SELECT timestamp, event_type, message
FROM event_log('2f1ed3ce-6581-422f-a51d-218cd2092450') -- pipeline id
ORDER BY timestamp DESC;
